Note: 

Both SwapRateMCSimulator and SwapRatePCAEstimator require historical swap rate time series data in a specific format.
If no explicit file path is provided to the constructor, a default CSV file will be used instead.

The default dataset is located in the "curve_data" folder of the project (see "eur_swap_ts.csv" for the expected format).

In the examples below, a file path is explicitly provided, although it points to the same data as the default.

This notebook demonstrates: 
- PCA for risk factor analysis of swap rates
- MC simulation of swap rate scenarios for generation of PnL scenarios and CVaR calculation 

In [21]:
from __future__ import annotations
import os 
from pathlib import Path

# It is assumed that the notebook is executed from within the project - either root or sub folder. 
CWD_PATH = Path.cwd()
SWAP_TIME_SERIES_CSV_PATH = ""

for parent in CWD_PATH.parents:
 if (parent / "src").exists():
     SWAP_TIME_SERIES_CSV_PATH = parent / "curve_data" / "eur_swap_ts.csv"
     break       
 else:
     SWAP_TIME_SERIES_CSV_PATH = CWD_PATH / "curve_data" / "eur_swap_ts.csv"

print(CWD_PATH)
print(SWAP_TIME_SERIES_CSV_PATH)   

C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype
C:\Users\mduba\Desktop\Programming\Python\alm_ir_hedging_prototype\curve_data\eur_swap_ts.csv


In [38]:
# imports 
from typing import Dict, Tuple
from datetime import datetime 
import numpy as np
import pandas as pd 
from curve_engine.swap_rate_pca_estimator import SwapRatePCAEstimator 
from curve_engine.swap_rate_mc_simulator import SwapRateMCSimulator

In [41]:
# run pca   
pca = SwapRatePCAEstimator.from_csv(SWAP_TIME_SERIES_CSV_PATH)

In [40]:
# PCA attributes 
pca.rates  
pca.shifts  
pca.cshifts        
pca.cov_mat 
pca.corr_mat
pca.pca_eigvals
pca.pca_eigvecs
pca.pca_loadings

,Node,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10
0,2Y,0.037079,-0.022772,-0.000398,0.001814,-0.004395,0.005655,-0.003042,0.002129,-0.000436,0.000473
1,3Y,0.045491,-0.026703,-0.013130,-0.000975,0.002572,-0.003222,0.001835,-0.000490,0.000216,-0.000096
2,5Y,0.045742,-0.010891,0.009882,0.001706,-0.000224,0.000182,0.000322,-0.002955,0.001464,-0.002390
3,7Y,0.045907,-0.005637,0.009236,0.001894,0.000343,-0.002297,0.001132,-0.001984,-0.000717,0.002866
4,10Y,0.048364,0.005342,0.001986,-0.007770,-0.000017,0.001231,0.000201,-0.000671,-0.002960,-0.000731
5,12Y,0.044917,0.004085,0.005536,0.002000,0.003648,-0.000884,0.002548,0.005546,-0.000101,-0.000557
6,15Y,0.045418,0.010095,-0.000408,-0.005973,0.000711,0.001070,-0.001614,0.000587,0.003343,0.000999
7,20Y,0.042114,0.011759,-0.002401,0.002588,-0.000848,-0.005459,-0.005815,0.000370,-0.000641,-0.000484
8,25Y,0.041514,0.016649,-0.005285,0.001759,-0.008146,-0.000543,0.003829,-0.000114,0.000291,-0.000127
9,30Y,0.041743,0.017471,-0.006459,0.004489,0.005248,0.005222,-0.000054,-0.002027,-0.000405,0.000127


In [42]:
# run mc simulation with scenarios for market/swap rates - 1000 scenarios, 1d shocks scaled to 30 day horizon
swap_rate_sim = SwapRateMCSimulator.from_csv(
  n_scenarios = 1000, 
  t_scale_factor = 30, 
  rng_seed = 1, 
  swap_ts_csv_path = SWAP_TIME_SERIES_CSV_PATH 
)

In [43]:
# MC attributes 
swap_rate_sim.rates
swap_rate_sim.shifts
swap_rate_sim.cshifts      
swap_rate_sim.cov_mat
swap_rate_sim.corr_mat
swap_rate_sim.rate_scenarios
swap_rate_sim.shift_scenarios

,Scenario,2Y,3Y,5Y,7Y,10Y,12Y,15Y,20Y,25Y,30Y
0,1,0.083849,0.183106,0.122018,0.076672,0.157540,0.122761,0.132179,0.131824,0.148810,0.171716
1,2,0.006896,0.065475,-0.074097,-0.085911,-0.129267,-0.078667,-0.118216,-0.112679,-0.176830,-0.138378
2,3,0.001976,-0.026722,0.146022,0.200332,-0.018559,0.019276,-0.059486,-0.028435,-0.066400,-0.103313
3,4,0.513853,0.475312,0.461639,0.494543,0.510036,0.495925,0.455028,0.323124,0.385627,0.377993
4,5,-0.297793,-0.415158,-0.303807,-0.328878,-0.343484,-0.308869,-0.316251,-0.332446,-0.273606,-0.232609
...,...,...,...,...,...,...,...,...,...,...,...
995,996,0.140752,0.089412,0.174410,0.150649,0.258524,0.149791,0.255350,0.199978,0.326549,0.285876
996,997,-0.466692,-0.394817,-0.562344,-0.500173,-0.587348,-0.490406,-0.507843,-0.559286,-0.627709,-0.367781
997,998,-0.064982,0.120847,-0.036450,-0.037313,-0.035590,-0.048425,-0.038113,0.013162,-0.021188,0.077208
998,999,0.000823,-0.087176,0.052070,0.038010,0.116607,0.084788,0.096018,0.088100,0.061058,0.086177
